In [91]:
import os

import keras
import numpy as np
import pandas as pd
from keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_PATH = "../../data/"
FEATURES = [
    "alcohol",
    "fixed acidity", 
    "volatile acidity",
    "citric acid", 
    "density",
    "residual sugar", 
    "chlorides", 
    "free sulfur dioxide", 
    "total sulfur dioxide", 
    "pH",
    "sulphates",
    # "quality",
    # "type",
    "is_white",
] 

In [ ]:
from pathlib import Path


def get_incremental_path(path_str: str) -> str:
    path = Path(path_str)
    
    if not path.exists():
        return str(path)
    
    parent = path.parent
    stem = path.stem
    suffix = path.suffix
    
    counter = 1
    while True:
        new_path = parent / f"{stem}_{counter}{suffix}"
        if not new_path.exists():
            return str(new_path)
        counter += 1

In [93]:
train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"), sep=";")
train['is_white'] = (train['type'] == 'white').astype(int)
train.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,is_white
count,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000,6714.000000
mean,7.251638,0.348436,0.317466,5.411871,0.057506,30.132559,114.576259,0.994789,3.219103,0.536120,10.458743,5.796098,0.734733
std,1.330423,0.168505,0.148842,4.739180,0.038175,17.619884,56.417138,0.003020,0.160737,0.157877,1.189933,0.883490,0.441508
min,3.800000,0.100000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987100,2.720000,0.220000,8.000000,3.000000,0.000000
25%,6.400000,0.200000,0.240000,1.800000,0.038000,16.000000,75.000000,0.992400,3.110000,0.430000,9.500000,5.000000,0.000000
50%,7.000000,0.300000,0.310000,3.000000,0.048000,28.000000,117.000000,0.995100,3.210000,0.510000,10.200000,6.000000,1.000000
75%,7.700000,0.400000,0.390000,8.000000,0.068000,41.000000,155.000000,0.997100,3.320000,0.600000,11.300000,6.000000,1.000000
max,15.900000,1.300000,1.660000,65.800000,0.611000,289.000000,440.000000,1.039000,4.010000,2.000000,14.900000,9.000000,1.000000


In [94]:
X = train[FEATURES].values
y = train["quality"].values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

In [95]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [96]:
inputs = keras.Input(shape=(len(FEATURES),))

x = layers.Dense(64, activation="relu")(inputs)
x = layers.BatchNormalization()(x)
# x = layers.Dropout(0.2)(x)

x = layers.Dense(32, activation="relu")(x)
x = layers.BatchNormalization()(x)
# x = layers.Dropout(0.2)(x)

x = layers.Dense(16, activation="relu")(x)

outputs = layers.Dense(1, activation="linear")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="wife")
model.summary()

Model: "wife"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 12)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,841 (15.00 KB)

 Trainable params: 3,649 (14.25 KB)

 Non-trainable params: 192 (768.00 B)

In [97]:
model.compile(
    loss=keras.losses.MeanSquaredError(),
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=["mean_absolute_error", "root_mean_squared_error"],
)


In [98]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=15, 
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        verbose=1
    )
]
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=32, epochs=100)

Epoch 1/100


168/168 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 6.4826 - mean_absolute_error: 1.9600 - root_mean_squared_error: 2.5461 - val_loss: 1.4593 - val_mean_absolute_error: 0.9697 - val_root_mean_squared_error: 1.2080
Epoch 2/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.8144 - mean_absolute_error: 0.7078 - root_mean_squared_error: 0.9024 - val_loss: 0.6489 - val_mean_absolute_error: 0.6262 - val_root_mean_squared_error: 0.8055
Epoch 3/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7558 - mean_absolute_error: 0.6804 - root_mean_squared_error: 0.8693 - val_loss: 0.8812 - val_mean_absolute_error: 0.7513 - val_root_mean_squared_error: 0.9387
Epoch 4/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7413 - mean_absolute_error: 0.6734 - root_mean_squared_error: 0.8610 - val_loss: 0.6083 - val_mean_absolute_error: 0.6100 - val_root_mean_squared_error: 0.7799
Epoch 5/100
168/168 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6955 - mean_absolute_error: 0.6471 - root_mean_squared_er

In [99]:
test_scores = model.evaluate(X_val, y_val, verbose=2)
print("Val loss:", test_scores[0])
print("Val accuracy:", test_scores[1])

42/42 - 0s - 1ms/step - loss: 0.5069 - mean_absolute_error: 0.5624 - root_mean_squared_error: 0.7120
Val loss: 0.5069035291671753
Val accuracy: 0.5624328255653381


In [100]:
model.save(get_incremental_path("../../models/wife.keras"))

### Inference

In [101]:
infer = pd.read_csv(os.path.join(DATA_PATH, "test.csv"), sep=";")
infer['is_white'] = (infer['type'] == 'white').astype(int)

In [102]:
X = infer[FEATURES]
y = model(X)

df = pd.DataFrame({"id": infer["id"], "quality": np.round(y[..., 0]).astype(int)})
df.to_csv(get_incremental_path("../../submit/wife.csv"), index=False)